In [ ]:
import json
import pandas as pd
import numpy as np
import os
import torch
from tirex import load_model


def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# ============================================================
# LOAD SPLITS
# ============================================================
path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
with open(path, "r") as f:
    dataset_days = json.load(f)

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation"
]

# ============================================================
# LOAD TIREX MODEL
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# safer on Windows:
model = load_model("NX-AI/TiRex", backend="torch", device=device)

# If the above works fine and you want, you can later try:
# model = load_model("NX-AI/TiRex", backend="torch", device=device, compile=True)

prediction_length = 96
max_context_length = 10000   # same spirit as your Chronos code

rmse_results = []

# ============================================================
# MAIN LOOP
# ============================================================
for country in countries:
    print("Processing country:", country)

    data_path = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [col for col in df.columns if col not in features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            # ------------------------------------------------
            # TRAINING CONTEXT
            # ------------------------------------------------
            s_train = df.loc[df.index < cutoff, household].dropna()

            # ground truth for forecast horizon
            y_true = df.loc[df.index >= cutoff, household].dropna().head(prediction_length)

            # skip if not enough train/test points
            if len(s_train) < 2:
                print(f"      Skipping {household}: not enough training history")
                continue

            if len(y_true) < prediction_length:
                print(f"      Skipping {household}: only {len(y_true)} future points available")
                continue

            # last context window
            context_values = s_train.tail(max_context_length).to_numpy(dtype=np.float32)

            # ------------------------------------------------
            # TIREX FORECAST
            # ------------------------------------------------
            try:
                quantiles, mean = model.forecast(
                    context=context_values,
                    prediction_length=prediction_length,
                    output_type="numpy"
                )

                # mean shape: (1, prediction_length)
                y_pred = mean[0]

            except Exception as e:
                print(f"      Error for {household}: {e}")
                continue

            # ------------------------------------------------
            # STORE PREDICTIONS
            # ------------------------------------------------
            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=y_true.index)

            predictions_df_all_households[household] = y_pred            


            # ------------------------------------------------
            # RMSE
            # ------------------------------------------------
            rmse = root_mean_squared_error(y_true.to_numpy(), y_pred)
            rmse_households.append(rmse)

        # ----------------------------------------------------
        # DAY SUMMARY
        # ----------------------------------------------------
        if len(rmse_households) == 0:
            avg_rmse_households = np.nan
        else:
            avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        # save predictions for this (country, day)
        output = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TiRexUnivar_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        predictions_df_all_households.to_csv(output, index=True)
        print("      Saved:", output)

# ============================================================
# SUMMARY TABLE
# ============================================================
rmse_df = pd.DataFrame(rmse_results)

print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

Using device: cpu
Processing country: Germany
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TiRexUnivar_pred_day1_Germany.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TiRexUnivar_pred_day2_Germany.csv
   Day: day3
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TiRexUnivar_pred_day3_Germany.csv
   Day: day4
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TiRexUnivar_pred_day4_Germany.csv
   Day: day5
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TiRexUnivar_pred_day5_Germany.csv
Processing country: Ireland
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TiRexUnivar_pred_day1_Ireland.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learn